## Baseline Thermal Inputs

The first Ansys model will use a conservative MOSFET heat load of **1.505 W** at the **10 A baseline operating point**. The ambient temperature is fixed at **25°C**.

The absolute IRFZ44N junction-temperature limit is **175°C**. A lower design target of **125°C** is retained as a safety-margin criterion.

For the simplified thermal model, the MOSFET heat is applied over a rectangular package/contact area of:

`15.8 mm × 10.0 mm`

This gives a heat-source area of:

`158 mm²`

The same heat-source dimensions must be used consistently in the analytical thermal-resistance calculation and in the Ansys geometry.

## Thermal Path

The assumed heat-flow path is:

`MOSFET junction → MOSFET case/tab → thermal interface material (TIM) → heat-sink base → heat-sink fins → ambient air`

The heat is generated at the MOSFET junction and conducted through the package to the case or metal tab. It then passes through the TIM into the heat-sink base, spreads through the heat sink and is finally transferred from the fins to the surrounding air by natural convection.

## No-Heat-Sink Reference Case

A **no-heat-sink case** is included as a reference to quantify the benefit of adding the TIM and heat sink.

Without a heat sink, the MOSFET rejects heat directly from its TO-220 package to the surrounding air.

**Thermal path:**

**Junction → MOSFET package → ambient air**

The junction temperature is estimated using the IRFZ44N junction-to-ambient thermal resistance:

**Tj,no heat sink = Ta + Ploss × RθJA**

The same conservative heat input of **1.505 W** and ambient temperature of **25°C** are used so that the no-heat-sink and heat-sink cases can be compared fairly.

In [1]:
import pandas as pd

# No-heat-sink reference calculation

ambient_temperature = 25.0       # °C
heat_input = 1.505               # W
r_theta_ja = 62.0                # °C/W 

maximum_junction_temperature = 175.0   # °C
recommended_junction_temperature = 125.0  # °C - project target

tj_no_heatsink = (
    ambient_temperature
    + heat_input * r_theta_ja
)

margin_to_maximum = (
    maximum_junction_temperature
    - tj_no_heatsink
)

margin_to_recommended = (
    recommended_junction_temperature
    - tj_no_heatsink
)

no_heatsink_results = pd.DataFrame({
    "Case": ["No heat sink"],
    "Heat input (W)": [heat_input],
    "Thermal resistance (°C/W)": [r_theta_ja],
    "Estimated junction temperature (°C)": [tj_no_heatsink],
    "Margin below maximum (°C)": [margin_to_maximum],
    "Margin below recommended target (°C)": [margin_to_recommended]
})

no_heatsink_results.round(2)

,Case,Heat input (W),Thermal resistance (°C/W),Estimated junction temperature (°C),Margin below maximum (°C),Margin below recommended target (°C)
0,No heat sink,1.5,62.0,118.31,56.69,6.69


## 1. Thermal-Resistance Path


The corresponding total thermal resistance is:

$$
R_{\theta,\mathrm{total}}
=
R_{\theta JC}
+
R_{\theta TIM}
+
R_{\theta HS}
+
R_{\theta conv}
$$

where:

- $R_{\theta JC}$ is the junction-to-case thermal resistance of the MOSFET.
- $R_{\theta TIM}$ is the thermal resistance through the thermal interface material.
- $R_{\theta HS}$ is the conduction resistance through the heat-sink base and into the fins.
- $R_{\theta conv}$ is the convection resistance from the exposed heat-sink surface to the surrounding air.

For this first simplified calculation, perfect contact is assumed between the MOSFET case, TIM and heat sink. Therefore, separate contact resistances at the case–TIM and TIM–heat-sink interfaces are neglected.

The heat-sink fins are included through the total exposed surface area used in the convection calculation. A separate fin resistance is not added in this initial model.

In [1]:
# Pre-FEA junction-temperature estimate
# Aluminium, copper and hybrid heat-sink constructions

import pandas as pd

# -------------------------------------------------
# Fixed electrical and MOSFET inputs
# -------------------------------------------------

P_loss = 1.505          # W, conservative baseline MOSFET heat load
T_ambient = 25.0        # degC
R_jc = 1.5              # degC/W, IRFZ44N junction-to-case resistance

# MOSFET/TIM contact area
source_width = 15.8e-3   # m
source_height = 10.0e-3  # m
A_contact = source_width * source_height

# -------------------------------------------------
# TIM: TGP 5000
# Values supplied by materials engineer
# -------------------------------------------------

t_tim = 1.5e-3          # m
k_tim = 5.0             # W/(m K)

# -------------------------------------------------
# Heat-sink materials
# Values supplied by materials engineer
# -------------------------------------------------

k_aluminium = 170.0     # W/(m K), aluminium 6061-T6
k_copper = 390.0        # W/(m K), C11000 copper

# -------------------------------------------------
# Heat-sink geometry and convection
# Replace these when final geometry is frozen
# -------------------------------------------------

t_base = 5.0e-3         # m, provisional heat-sink base thickness
h = 10.0                 # W/(m^2 K), provisional natural convection coefficient
A_surface = 0.0533      # m^2, total exposed base and fin surface area

# -------------------------------------------------
# Common thermal resistances
# -------------------------------------------------

R_tim = t_tim / (k_tim * A_contact)
R_convection = 1.0 / (h * A_surface)

# -------------------------------------------------
# Function for calculating each construction
# -------------------------------------------------

def calculate_case(case_name, base_conductivity, fin_material):
    """
    Calculate the simplified pre-FEA thermal resistance and junction
    temperature for one heat-sink construction.

    In this simple model, the heat-sink conduction term represents
    conduction through the base. Fin material is recorded for reference,
    while the fins are represented through the exposed convection area.
    """

    R_base = t_base / (base_conductivity * A_contact)

    R_total = (
        R_jc
        + R_tim
        + R_base
        + R_convection
    )

    temperature_rise = P_loss * R_total
    T_junction = T_ambient + temperature_rise

    return {
        "Construction": case_name,
        "Base material": (
            "Aluminium 6061-T6"
            if base_conductivity == k_aluminium
            else "C11000 copper"
        ),
        "Fin material": fin_material,
        "R_jc (degC/W)": R_jc,
        "R_TIM (degC/W)": R_tim,
        "R_base (degC/W)": R_base,
        "R_convection (degC/W)": R_convection,
        "R_total (degC/W)": R_total,
        "Temperature rise (degC)": temperature_rise,
        "Estimated Tj (degC)": T_junction,
    }

# -------------------------------------------------
# No-heat-sink reference case
# -------------------------------------------------

R_ja = 62.0  # degC/W, IRFZ44N junction-to-ambient thermal resistance

temperature_rise_no_heatsink = P_loss * R_ja
T_junction_no_heatsink = T_ambient + temperature_rise_no_heatsink

no_heatsink_case = {
    "Construction": "No heat sink",
    "Base material": "None",
    "Fin material": "None",
    "R_jc (degC/W)": None,
    "R_TIM (degC/W)": None,
    "R_base (degC/W)": None,
    "R_convection (degC/W)": None,
    "R_total (degC/W)": R_ja,
    "Temperature rise (degC)": temperature_rise_no_heatsink,
    "Estimated Tj (degC)": T_junction_no_heatsink,
}

# -------------------------------------------------
# Define the three cooling constructions
# -------------------------------------------------

results = [
    no_heatsink_case,
    calculate_case(
        case_name="All aluminium",
        base_conductivity=k_aluminium,
        fin_material="Aluminium 6061-T6",
    ),
    calculate_case(
        case_name="All copper",
        base_conductivity=k_copper,
        fin_material="C11000 copper",
    ),
    calculate_case(
        case_name="Hybrid",
        base_conductivity=k_copper,
        fin_material="Aluminium 6061-T6",
    ),
]

results_df = pd.DataFrame(results)

# Round numerical results for clearer display
numeric_columns = results_df.select_dtypes(include="number").columns
results_df[numeric_columns] = results_df[numeric_columns].round(3)

print(f"Contact area: {A_contact * 1e6:.1f} mm^2")
print(f"TGP 5000 TIM resistance: {R_tim:.3f} degC/W")
print(f"Convection resistance: {R_convection:.3f} degC/W")
print()

display(results_df)

Contact area: 158.0 mm^2
TGP 5000 TIM resistance: 1.899 degC/W
Convection resistance: 1.876 degC/W



,Construction,Base material,Fin material,R_jc (degC/W),R_TIM (degC/W),R_base (degC/W),R_convection (degC/W),R_total (degC/W),Temperature rise (degC),Estimated Tj (degC)
0,No heat sink,None,None,NaN,NaN,NaN,NaN,62.000,93.310,118.310
1,All aluminium,Aluminium 6061-T6,Aluminium 6061-T6,1.5,1.899,0.186,1.876,5.461,8.219,33.219
2,All copper,C11000 copper,C11000 copper,1.5,1.899,0.081,1.876,5.356,8.061,33.061
3,Hybrid,C11000 copper,Aluminium 6061-T6,1.5,1.899,0.081,1.876,5.356,8.061,33.061


## Analytical vs Ansys Baseline Comparison

The first steady-state Ansys simulation produced a maximum model temperature of approximately **30.63°C** for the baseline heat-sink case, with Aluminium.

![Baseline Ansys temperature result](images/Ansys_1.png)

Because the simplified Ansys model does not explicitly represent the internal silicon junction of the IRFZ44N, the maximum simulated temperature is treated as an approximate MOSFET case/package temperature. The junction temperature is therefore estimated by adding the junction-to-case temperature rise:

**Tj = Tcase + Ploss × RθJC**

Using:

- **Tcase,FEA = 30.63°C**
- **Ploss = 1.505 W**
- **RθJC = 1.5°C/W**

the estimated junction temperature from the Ansys result is:

**Tj,FEA = 30.63 + (1.505 × 1.5)**

**Tj,FEA ≈ 32.88°C**

The analytical thermal-resistance model predicts a junction temperature of approximately **33.22°C** using the same baseline heat input, ambient temperature and convection assumptions.

The difference between the two junction-temperature estimates is therefore:

**33.22°C − 32.88°C = 0.34°C**

This corresponds to a difference of approximately **1.0%**.

The analytical and FEA results therefore show very close agreement. The remaining small difference is reasonable because the analytical model represents the thermal path as a one-dimensional series resistance network, while Ansys models three-dimensional heat conduction and the actual heat-sink geometry.

The original analytical estimate was higher because the assumed heat-sink convection surface area did not match the area used in Ansys. After updating the analytical model to use the Ansys convection area of approximately **0.0533 m²** and a convection coefficient of **10 W/m²K**, the analytical and FEA results became closely aligned.

| Method | Temperature |
|---|---:|
| Analytical junction-temperature estimate | 33.22°C |
| Ansys maximum case/package temperature | 30.63°C |
| Ansys-derived junction-temperature estimate | 32.88°C |
| Absolute difference | 0.34°C |
| Percentage difference | 1.0% |

## No-Heat-Sink Reference

The no-heat-sink case is kept separate from the cooled FEA comparison and is calculated using the datasheet junction-to-ambient thermal resistance.

Using:

- **Ta = 25°C**
- **Ploss = 1.505 W**
- **RθJA = 62°C/W**

the estimated no-heat-sink junction temperature is:

**Tj,no heat sink = 25 + (1.505 × 62)**

**Tj,no heat sink ≈ 118.31°C**

The no-heat-sink result is then compared with both the analytical and FEA-derived cooled junction-temperature estimates.

### Comparison with analytical cooled estimate

The analytical thermal-resistance model predicts a cooled junction temperature of approximately:

**Tj,analytical ≈ 33.22°C**

Therefore, the predicted temperature reduction produced by the baseline TIM and heat sink is:

**Temperature reduction = 118.31°C − 33.22°C**

**Temperature reduction ≈ 85.09°C**

### Comparison with FEA-derived cooled estimate

The Ansys simulation produced a maximum case/package temperature of approximately **30.63°C**.

After accounting for the junction-to-case temperature rise, the estimated junction temperature is:

**Tj,FEA ≈ 32.88°C**

Therefore, the predicted temperature reduction using the FEA-derived junction estimate is:

**Temperature reduction = 118.31°C − 32.88°C**

**Temperature reduction ≈ 85.43°C**

| Comparison method | Estimated cooled junction temperature | Temperature reduction from no heat sink |
|---|---:|---:|
| Analytical thermal-resistance model | 33.22°C | 85.09°C |
| FEA-derived junction estimate | 32.88°C | 85.43°C |

Both methods therefore predict a junction-temperature reduction of approximately **85°C** when the baseline TIM and heat sink are used.

This demonstrates the strong thermal benefit of dedicated cooling under the baseline MOSFET heat load.

The close agreement between the analytical and FEA-derived cooled temperatures also supports the consistency of the two thermal modelling approaches.

The no-heat-sink result is an analytical reference only. Junction-to-ambient thermal resistance depends strongly on package mounting, PCB layout and surrounding airflow, so it should not be treated as directly equivalent to the detailed Ansys heat-sink model.

## Assumptions

The thermal resistances are represented as a one-dimensional series network from the MOSFET junction to ambient air. Junction-to-case resistance is taken from the IRFZ44N datasheet.

TIM and heat-sink conduction resistances are calculated using **R = t/(kA)**. Convection resistance is estimated using **R = 1/(hA)**.

Perfect contact is initially assumed between the MOSFET case, TIM and heat sink, so separate contact resistances are neglected. Radiation, heat loss through the MOSFET leads and three-dimensional heat spreading are also neglected. These effects will be represented more accurately in the Ansys model.

The no-heat-sink case is calculated separately using the datasheet junction-to-ambient thermal resistance, **RθJA**. It does not use the junction-to-case, TIM, heat-sink base and convection resistance network applied to the heat-sink cases.

## Heat-Input Consistency Check

The analytical thermal-resistance model and the Ansys FEA model use the
same baseline MOSFET heat input of **1.505 W**.

In the analytical model, this value is entered directly as the MOSFET
power loss.

In Ansys, the same heat input is applied either as a total heat load of
**1.505 W** or as an equivalent heat flux of approximately
**9525 W/m²** over the selected 15.8 mm × 10.0 mm source face.

Only one of these Ansys heat-input methods is used to avoid
double-counting the MOSFET power loss.

## Heat-Load Cases

Five thermal heat-load cases are prepared for the thermal validation and later materials comparison.

### Electrically Derived Cases

The 5 A, 10 A and 20 A cases are calculated from the buck-converter MOSFET loss model using the conservative datasheet maximum on-resistance of **0.0175 ohm at 25°C**.

The resulting heat loads are:

| Operating case | MOSFET heat input |
|---|---:|
| 5 A | 0.534 W |
| 10 A | 1.505 W |
| 20 A | 4.760 W |

These cases represent low, baseline and high electrical operating conditions.

### Imposed Thermal-Stress Cases

Two additional thermal loads are defined independently of the electrical operating points:

| Thermal-stress case | Heat input |
|---|---:|
| High thermal stress | 10 W |
| Extreme thermal stress | 15 W |

The 10 W and 15 W cases are not claimed to correspond directly to a specific converter current. They are imposed thermal-stress cases used to investigate how the cooling system and heat-sink material behave under much higher heat dissipation.

### Overall Cases

| Case                   | Scenario type          | Load current | Heat input |
| ---------------------- | ---------------------- | -----------: | ---------: |
| Low load               | Electrically derived   |          5 A |    0.534 W |
| Baseline               | Electrically derived   |         10 A |    1.505 W |
| High load              | Electrically derived   |         20 A |    4.760 W |
| High thermal stress    | Thermal stress         |          N/A |   10.000 W |
| Extreme thermal stress | Thermal stress         |          N/A |   15.000 W |
| No heat sink           | Reference              |          N/A |    1.505 W |

All thermal simulations use the same MOSFET heat-source area of **0.000158 m²**. Only the heat input is changed when investigating the effect of thermal load.

The same geometry, ambient temperature, convection coefficient, TIM and material properties should be retained when comparing these heat loads so that the effect of increasing heat input can be isolated.

## Junction Temperature Calculation

ANSYS provides the temperature at the MOSFET case / heat-source interface.

### ANSYS Case Temperature Results

The maximum case temperature, \(T_c\), obtained from ANSYS for each heat input is shown below.

| Heat Input (W) | Aluminium \(T_c\) (°C) | Copper \(T_c\) (°C) |
|---:|---:|---:|
| 0.534 | 26.997 | 26.927 |
| 1.505 | 30.627 | 30.430 |
| 4.760 | 42.798 | 42.175 |
| 10.0 | 62.390 | 61.082 |
| 15.0 | 81.095 | 79.124 |

### Calculating Junction Temperature

Again, the junction temperature is estimated using:

Tj = Tc + P × RθJC

where:
- Tj = junction temperature (°C)
- Tc = ANSYS case temperature (°C)
- P = MOSFET power loss (W)
- RθJC = junction-to-case thermal resistance (°C/W)

For the IRFZ44N, RθJC = 1.5 °C/W.

In [2]:
import pandas as pd

power = [0.534, 1.505, 4.760, 10.0, 15.0]
current = ["5 A", "10 A", "20 A", "Stress case", "Stress case"]
aluminium_tc = [26.997, 30.627, 42.798, 62.39, 81.095]
copper_tc = [26.927, 30.43, 42.175, 61.082, 79.124]


R_theta_JC = 1.5  # °C/W

aluminium_tj = [
    tc + p * R_theta_JC
    for tc, p in zip(aluminium_tc, power)
]

copper_tj = [
    tc + p * R_theta_JC
    for tc, p in zip(copper_tc, power)
]

results = pd.DataFrame({
    "Power (W)": power,
    "Current": current,
    "Aluminium Tc (°C)": aluminium_tc,
    "Aluminium Tj (°C)": aluminium_tj,
    "Copper Tc (°C)": copper_tc,
    "Copper Tj (°C)": copper_tj
})

results.round(3)



,Power (W),Current,Aluminium Tc (°C),Aluminium Tj (°C),Copper Tc (°C),Copper Tj (°C)
0,0.534,5 A,26.997,27.798,26.927,27.728
1,1.505,10 A,30.627,32.884,30.430,32.688
2,4.760,20 A,42.798,49.938,42.175,49.315
3,10.000,Stress case,62.390,77.390,61.082,76.082
4,15.000,Stress case,81.095,103.595,79.124,101.624


### Interpretation

The junction temperature increases with MOSFET power dissipation for both
heatsink materials. Copper produces slightly lower temperatures than aluminium,
although the difference remains relatively small.

At 15 W, the calculated junction temperatures are approximately 103.6 °C for
aluminium and 101.6 °C for copper, corresponding to a reduction of around
2 °C when copper is used.

This suggests that while heatsink thermal conductivity affects the result,
other thermal resistances, particularly convection from the heatsink to the
surrounding air, have a significant influence on the overall thermal performance.